# 06b · Run tool on records crawled directly (gbcrawler) → extract gene(s)

**Purpose.** End-to-end trên record **tự crawl từ GenBank** bằng `gbcrawler`, rồi chạy tool trích gene — chứng minh luồng từ accession thô → annotation, khác 06a (chạy trên file có sẵn).

## Inputs

- `gbcrawler/fetch.py` (cần network tới NCBI)
- một danh sách accession / query term

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
for c in [ROOT, *ROOT.parents]:
    if (c / "app" / "src").exists():
        ROOT = c; break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
UNIT_DIR = ROOT / "app" / "validation" / "06_run_tool_extract"
OUT = UNIT_DIR / "outputs" / "crawled"; OUT.mkdir(parents=True, exist_ok=True)

## Crawl

> ⚠️ **TODO**: gọi `gbcrawler.fetch` để tải record về `OUT` (cần network — không chạy được trong sandbox). Xem `gbcrawler/README.md` cho API/CLI. Ví dụ dạng: `python -m gbcrawler <accessions/term> --out OUT`.

In [ ]:
# import subprocess
# subprocess.run([sys.executable, "-m", "gbcrawler", "--out", str(OUT), "--query", "PRRSV ORF5"], check=True)
crawled_gb = OUT / "crawled.gb"   # TODO: đường dẫn file .gb sau khi crawl

## Run tool → extract gene(s)

In [ ]:
from app.src.io.genbank_parser import load_genbank_records
from app.src.features.annotation_strategy import get_strategy
from app.src.features.direct_extractor import direct_extract_with_alias
from app.src.lifting.tblastn_lifter import lift_all_tblastn
from app.validation._shared.validation_utils import load_reference_bundle, lifted_to_rows
import pandas as pd

REF = ROOT / "app" / "data" / "PRRS" / "PRRS_ref_test.gb"  # TODO: ref khớp virus đã crawl
bundle = load_reference_bundle(REF)
rows = []
for rec in load_genbank_records(crawled_gb):
    strat, ft = get_strategy(rec, bundle["alias_lookup"])
    lifted = (direct_extract_with_alias(rec, ft, bundle["features"], bundle["alias_lookup"])
              if strat == "direct"
              else lift_all_tblastn(ref_features=bundle["features"], ref_record=bundle["record"],
                                    query_record=rec, validate_codons=(bundle["feature_type"]=="CDS")))
    for r in lifted_to_rows(rec.id, lifted, strat):
        rows.append(r)
extracted = pd.DataFrame(rows)
extracted.to_csv(OUT / "crawled_extractions.tsv", sep="\t", index=False)
extracted.head()

## Interpretation

> ⚠️ **TODO**: báo tool gán được gene cho record crawl thô, kể cả chưa annotation.